# GPT-2 Transformer via UA Engine DSL

Builds a 2-layer transformer using the UA Engine DSL: morphisms declare ops, paths compose them, and `arch` declares the block structure with `iterate=layers`. `run_algebra` folds the tree bottom-up for a parallel batch forward pass.

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import engine

print(f"engine loaded from: {engine.__file__}")

engine loaded from: /home/scanbot/ua_tensors/engine/__init__.py


In [9]:
EPS = 1e-5

class Ops:
    @staticmethod
    def join_fn(x, y):
        return x

    @staticmethod
    def compile_einsum(eq):
        return eq

    # Parameterized linear projection — one op, many instantiations via proj[prefix]
    @staticmethod
    def proj(eq, x, y, temp=0.0, prefix=None):
        return x @ y[f'{prefix}_W'] + y[f'{prefix}_b']

    # Parameterized layer norm — ln[prefix] reads prefix_g / prefix_b
    @staticmethod
    def ln(eq, x, y, temp=0.0, prefix=None):
        g = y.get(f'{prefix}_g', np.ones(x.shape[-1]))
        b = y.get(f'{prefix}_b', np.zeros(x.shape[-1]))
        mu  = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1,  keepdims=True)
        return ((x - mu) / np.sqrt(var + EPS)) * g + b

    @staticmethod
    def score(eq, q, y, temp=0.0):
        # proj[k] and proj[v] are injected into y by the [kv] augment.
        # Fan branch names become the dict keys: 'proj[k]' and 'proj[v]'.
        K = y['proj[k]']
        scale = y.get('scale', K.shape[-1] ** -0.5)
        s = q @ K.T * scale
        if y.get('mask') is not None:
            s = s + y['mask']
        return s

    @staticmethod
    def softmax(eq, x, y=None, temp=0.0):
        # arity unary — y unused
        x = x - x.max(axis=-1, keepdims=True)
        e = np.exp(x)
        return e / e.sum(axis=-1, keepdims=True)

    @staticmethod
    def mix(eq, probs, y, temp=0.0):
        return probs @ y['proj[v]']

    @staticmethod
    def gelu(eq, x, y=None, temp=0.0):
        # arity unary — y unused
        return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

ops = Ops()
print("ops ready:", [m for m in dir(ops) if not m.startswith('_')])

ops ready: ['compile_einsum', 'gelu', 'join_fn', 'ln', 'mix', 'proj', 'score', 'softmax']


In [10]:
DSL_SOURCE = """
semiring attn:
    contract = ops.join_fn
    compiler = ops.compile_einsum

sort model, h, scores, probs, ff

# Parameterized linear projection — one op, two sort signatures.
# proj[prefix]     : model -> h    (Q, K, V, FFN-up)   equation "sd,dh->sh"
# proj_out[prefix] : h -> model    (attn-out, FFN-down) equation "sh,hd->sd"
# Both dispatch to ops.proj; prefix selects weight keys (e.g. 'q' -> q_W, q_b).
morphism proj[prefix]     : model -> h      via "sd,dh->sh"    op ops.proj
morphism proj_out[prefix] : h -> model      via "sh,hd->sd"    op ops.proj

# Parameterized layer norm
morphism ln[prefix]   : model -> model   via "sd->sd"        op ops.ln

# Unique ops
morphism score     : h -> scores      via "sh,th->st"    op ops.score
morphism normalize : scores -> probs  via "st->st"       op ops.softmax  arity unary
morphism mix       : probs -> h       via "st,th->sh"    op ops.mix
morphism act       : h -> h           via "sh->sh"       op ops.gelu     arity unary

# [kv] augment: runs proj[k] & proj[v], merges outputs into y as 'proj[k]'/'proj[v]'
fan kv = proj[k] & proj[v]  merge dict

path read = proj[q] score normalize mix proj_out[o]
path attn = ln[1] [kv] read  residual
path ffn  = ln[2] proj[up] act proj_out[down]  residual

arch Transformer:
    cases:
        input: leaf  data=1  cell=identity
        block: node  data=1  morphisms=attn ffn  iterate=layers
"""

print(f"DSL source: {len(DSL_SOURCE)} chars")

DSL source: 1390 chars


The `proj[prefix]` template syntax declares one morphism that gets instantiated with different prefixes. `proj[q]` reads `y['q_W']` and `y['q_b']`; `proj_out[o]` reads `y['o_W']` and `y['o_b']`. Same computation (`x @ y[f'{prefix}_W'] + y[f'{prefix}_b']`), different weight keys. Two templates (`proj` and `proj_out`) handle the two sort signatures (into hidden space vs back to model space), reducing 8 projection ops to 1 Python function.

In [11]:
arch = engine.compile(DSL_SOURCE, {'ops': ops})

print("Compiled paths:", sorted(arch.paths.keys()))
print()
print(arch.explain('attn'))

ValueError: Fan 'kv': branch 'proj[k]' is not a declared morphism or path

In [ ]:
# Hyperparameters
D, H, FF, VOCAB, SEQ, LAYERS = 16, 4, 32, 20, 4, 2

rng = np.random.default_rng(42)
N   = lambda shape: rng.normal(0.0, 0.02, shape)

def make_layer(d, h, ff):
    return {
        'q_W': N((d, h)), 'q_b': np.zeros(h),
        'k_W': N((d, h)), 'k_b': np.zeros(h),
        'v_W': N((d, h)), 'v_b': np.zeros(h),
        'o_W': N((h, d)), 'o_b': np.zeros(d),
        'up_W': N((d, ff)), 'up_b': np.zeros(ff),
        'down_W': N((ff, d)), 'down_b': np.zeros(d),
        '1_g': np.ones(d),  '1_b': np.zeros(d),   # ln[1]
        '2_g': np.ones(d),  '2_b': np.zeros(d),   # ln[2]
    }

layers     = [make_layer(D, H, FF) for _ in range(LAYERS)]
tok_embed  = N((VOCAB, D))
pos_enc    = N((64, D))
causal_mask = np.triu(np.full((SEQ, SEQ), -1e9), 1)
layer_bundles = [{**w, 'mask': causal_mask} for w in layers]

# Forward pass
token_ids = np.array([3, 7, 1, 15])
x0        = tok_embed[token_ids] + pos_enc[:SEQ]

interp = arch.interpreter('Transformer', params={}, temp=0.0)
result = interp.run_algebra(x0, layers=layer_bundles)

logits = result @ tok_embed.T
print(f"input:  {x0.shape}  output: {result.shape}  logits: {logits.shape}")
print("\nTop-1 prediction per position:")
for pos in range(SEQ):
    t = logits[pos].argmax()
    print(f"  pos {pos}: token {t:3d}  logit={logits[pos, t]:.4f}")

In [ ]:
def manual_ln(x, w, prefix):
    g = w.get(f'{prefix}_g', np.ones(x.shape[-1]))
    b = w.get(f'{prefix}_b', np.zeros(x.shape[-1]))
    mu, var = x.mean(axis=-1, keepdims=True), x.var(axis=-1, keepdims=True)
    return ((x - mu) / np.sqrt(var + EPS)) * g + b

def manual_attn(x, w, mask):
    xn = manual_ln(x, w, '1')
    Q = xn @ w['q_W'] + w['q_b']
    K = xn @ w['k_W'] + w['k_b']
    V = xn @ w['v_W'] + w['v_b']
    sc = Q @ K.T * (H ** -0.5) + mask
    sc -= sc.max(axis=-1, keepdims=True)
    probs = np.exp(sc) / np.exp(sc).sum(axis=-1, keepdims=True)
    return x + (probs @ V) @ w['o_W'] + w['o_b']

def manual_ffn(x, w):
    xn = manual_ln(x, w, '2')
    h = xn @ w['up_W'] + w['up_b']
    h = 0.5 * h * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (h + 0.044715 * h**3)))
    return x + h @ w['down_W'] + w['down_b']

def manual_block(x, w, mask):
    return manual_ffn(manual_attn(x, w, mask), w)

x_ref = x0.copy()
for w in layers:
    x_ref = manual_block(x_ref, w, causal_mask)

np.testing.assert_allclose(result, x_ref, atol=1e-12)
print("Engine output matches manual numpy forward pass.")

In [ ]:
mod = arch.module
print(f"Module: {mod.namespace.value}, {len(mod.definitions)} definitions")

g = arch.graph
print(f"Graph: {len(g.bound_terms)} terms, {len(g.primitives)} primitives")